In [1]:
!pip install selenium pyarrow lxml requests beautifulsoup4 pandas
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import os
import glob
import pickle
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
}

YEAR = 2024
SAVE_DIR = rf'C:\Users\lcsse\Desktop\STAGE_GENT\scrapping\data\pcs{YEAR}'
os.makedirs(SAVE_DIR, exist_ok=True)

dates = []
list_of_result_dfs = []
races = []
data_by_month = {}


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


/Users/arthurdeletang/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def is_page_not_found(response_text):
    return "Page not found" in response_text

def process_results_page(response_text, url):
    soup = BeautifulSoup(response_text, 'lxml')

    # Date
    try:
        date_tag = soup.select_one('div.value')
        date = date_tag.get_text(strip=True)
    except Exception:
        date = None
    dates.append(date)

    # Classification
    try:
        classif = None
        for li in soup.find_all('li'):
            if 'Classification:' in li.get_text():
                classif = li.get_text(strip=True).replace('Classification:', '').strip()
                break
    except Exception:
        classif = None

    # Startlist quality score
    try:
        startlist_quality = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Startlist quality score:' in text:
                match = re.search(r'Startlist quality score:\s*(\d+)', text)
                if match:
                    startlist_quality = int(match.group(1))
                break
    except Exception:
        startlist_quality = None

    # Average temperature
    try:
        avg_temperature = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Avg. temperature:' in text:
                match = re.search(r'Avg\. temperature:\s*([\d.]+)', text)
                if match:
                    avg_temperature = float(match.group(1))
                break
    except Exception:
        avg_temperature = None

    # Won how
    try:
        won_how = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Won how:' in text:
                won_how = text.split('Won how:')[-1].strip()
                break
    except Exception:
        won_how = None

    tables = soup.find_all('table')
    if tables:
        dfs = pd.read_html(str(tables))
        if dfs:
            df = dfs[0]
            df['date'] = date
            df['classification'] = classif
            df['startlist_quality'] = startlist_quality
            df['avg_temperature'] = avg_temperature
            df['won_how'] = won_how
            list_of_result_dfs.append(df)
            races.append(url)

def scrape_base_url(base_url):
    def has_results_table(response_text):
        soup = BeautifulSoup(response_text, 'lxml')
        return len(soup.find_all('table')) > 0

    result_url = base_url + "/result"
    response = requests.get(result_url, headers=headers)
    time.sleep(1)
    if has_results_table(response.text):
        process_results_page(response.text, result_url)

    stage = 1
    while True:
        stage_url = f"{base_url}/stage-{stage}"
        response = requests.get(stage_url, headers=headers)
        time.sleep(1)
        if has_results_table(response.text):
            process_results_page(response.text, stage_url)
            stage += 1
        else:
            break

    print(f"✅ {base_url}")

def scrape_urls(url_list):
    for base_url in url_list:
        scrape_base_url(base_url)

def clean_race_url(url):
    url = url.replace('/result', '')
    url = re.sub(r'/stage-\d+', '', url)
    return url

def save_month(month):
    for race, df, date in zip(data_by_month[month]['races'], data_by_month[month]['dfs'], data_by_month[month]['dates']):
        try:
            df['race_url'] = race
            df['date'] = date
            if '/stage-' in race:
                race_type = 'stage_' + race.split('/stage-')[-1]
            elif '/result' in race:
                race_type = 'result'
            else:
                race_type = 'one_day'
            race_name = race.replace('https://www.procyclingstats.com/race/', '')
            race_name = re.sub(r'/stage-\d+', '', race_name)
            race_name = race_name.replace('/result', '').replace('/', '_')
            clean_date = date.replace(' ', '_') if date else 'no_date'
            if 'classification' in df.columns and df['classification'].iloc[0]:
                classif = str(df['classification'].iloc[0]).strip()
            else:
                classif = 'no_classif'
            filename = f"{clean_date}__{race_type}__{race_name}__{classif}.csv"
            df.to_csv(os.path.join(SAVE_DIR, filename), index=False)
        except Exception as e:
            print(f"❌ {race}: {e}")
    print(f"✅ Mois {month}: {len(data_by_month[month]['races'])} fichiers sauvegardés")

In [3]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2024&month=1&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour janvier 2024")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[1] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(1)
driver.quit()

✅ 34 URLs trouvées pour janvier 2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-australia-itt/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-australia/2024


...

✅ https://www.procyclingstats.com/race/alula-tour/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is depre

✅ https://www.procyclingstats.com/race/etoile-de-besseges/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is depre

✅ https://www.procyclingstats.com/race/vuelta-a-la-comunidad-valenciana/2024
Total brut: 72 | Uniques: 34 | Manquantes: 0
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2024/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2024/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2024/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2024/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2024/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2024/stage-6: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-tropicale-amissa-bongo/2024/stage-7: single positional indexer is out-of-bounds
❌ https://www.procycling

In [4]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2024&month=2&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour fevrier 2024")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[2] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(2)
driver.quit()

✅ 33 URLs trouvées pour fevrier 2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-ecuador-itt/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-south-africa-itt/2024


...

✅ https://www.procyclingstats.com/race/gp-samyn/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/trofeo-laigueglia/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/trofej-umag/2024
Total brut: 72 | Uniques: 33 | Manquantes: 0
❌ https://www.procyclingstats.com/race/ruta-del-sol/2024/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ruta-del-sol/2024/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ruta-del-sol/2024/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ruta-del-sol/2024/stage-5: single positional indexer is out-of-bounds
✅ Mois 2: 72 fichiers sauvegardés


In [5]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2024&month=3&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour mars 2024")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[3] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(3)
driver.quit()

✅ 52 URLs trouvées pour mars 2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/strade-bianche/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/grand-prix-criquielion/2024


...

✅ https://www.procyclingstats.com/race/gran-premio-miguel-indurain/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/bueng-si-fai-international-road-race/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/ronde-van-vlaanderen/2024
Total brut: 92 | Uniques: 52 | Manquantes: 0
❌ https://www.procyclingstats.com/race/per-sempre-alfredo/2024/result: single positional indexer is out-of-bounds
✅ Mois 3: 92 fichiers sauvegardés


In [6]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2024&month=4&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour avril 2024")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[4] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(4)
driver.quit()

✅ 45 URLs trouvées pour avril 2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is depre

✅ https://www.procyclingstats.com/race/itzulia-basque-country/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is depre

✅ https://www.procyclingstats.com/race/tour-of-thailand/2024


...

✅ https://www.procyclingstats.com/race/nc-panama/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-united-arab-emirates/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is depre

✅ https://www.procyclingstats.com/race/tour-du-benin/2024
Total brut: 116 | Uniques: 45 | Manquantes: 0
❌ https://www.procyclingstats.com/race/tour-of-thessaly/2024/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-thessaly/2024/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-thessaly/2024/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-a-formosa-internacional/2024/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-a-formosa-internacional/2024/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-a-formosa-internacional/2024/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-a-formosa-internacional/2024/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-tu

In [7]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2024&month=5&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour mai 2024")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[5] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(5)
driver.quit()

✅ 69 URLs trouvées pour mai 2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/eschborn-frankfurt/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/omloop-van-het-waasland/2024


...

✅ https://www.procyclingstats.com/race/tour-of-malopolska/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is depre

✅ https://www.procyclingstats.com/race/tour-du-cameroun/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is depre

✅ https://www.procyclingstats.com/race/tour-du-maroc/2024
Total brut: 182 | Uniques: 68 | Manquantes: 1
❌ https://www.procyclingstats.com/race/tour-de-catamarca-internacional/2024/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-catamarca-internacional/2024/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-catamarca-internacional/2024/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-catamarca-internacional/2024/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/kozagawa-city-international-road-race/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/jelajah-cycling-series-surakarta/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-kirikkale/2024/stage-1: single positional indexer is out-of-bounds
❌ https:/

In [8]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2024&month=6&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour juin 2024")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[6] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(6)
driver.quit()

✅ 201 URLs trouvées pour juin 2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/trofeo-alcide-degasperi/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/heist-op-den-berg/2024


...

✅ https://www.procyclingstats.com/race/nc-peru/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-morocco/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-ivory-coast/2024
Total brut: 298 | Uniques: 201 | Manquantes: 0
❌ https://www.procyclingstats.com/race/trofeo-alcide-degasperi/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-libya/2024/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-libya/2024/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-libya/2024/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-libya/2024/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-libya/2024/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-libya/2024/stage-6: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-libya/2024/stage-7: single positional indexer is out-of-bounds
❌ htt

In [9]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2024&month=7&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour juillet 2024")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[7] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(7)
driver.quit()

✅ 43 URLs trouvées pour juillet 2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-iran/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/trofeo-citta-di-brescia/2024


...

✅ https://www.procyclingstats.com/race/national-championships-seychelles-me-itt/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/puchar-ministra-obrony-narodowej/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/national-championships-seychelles-me-road-race/2024
Total brut: 84 | Uniques: 43 | Manquantes: 0
❌ https://www.procyclingstats.com/race/tour-of-austria/2024/stage-6: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/la-maurienne/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/giro-di-reggio-calabria/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/settimana-ciclista-lombarda/2024/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/settimana-ciclista-lombarda/2024/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/settimana-ciclista-lombarda/2024/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-des-pays-de-savoie/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/

In [11]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2024&month=8&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour aout 2024")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[8] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(8)
driver.quit()

✅ 42 URLs trouvées pour aout 2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/olympic-games/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/cupa-max-ausnit/2024


...

✅ https://www.procyclingstats.com/race/muur-classic-geraardsbergen/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is depre

✅ https://www.procyclingstats.com/race/tour-of-samsun/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/in-the-footsteps-of-the-romans/2024
Total brut: 240 | Uniques: 42 | Manquantes: 0
❌ https://www.procyclingstats.com/race/national-championships-peru-wj-road-race/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/memorial-henryka-lasaka/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/memorial_jana_magiery_2022/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-borneo/2024/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-borneo/2024/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-borneo/2024/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/gp-stad-zottegem/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/baltic-chain-tour/

In [12]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2024&month=9&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour septembre 2024")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[9] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(9)
driver.quit()

✅ 48 URLs trouvées pour septembre 2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/maryland-cycling-classic/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gp-kranj/2024


...

✅ https://www.procyclingstats.com/race/oita-urban-classic/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/world-championship/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is depre

✅ https://www.procyclingstats.com/race/tour-de-langkawi/2024
Total brut: 94 | Uniques: 47 | Manquantes: 1
❌ https://www.procyclingstats.com/race/maryland-cycling-classic/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-kosovo/2024/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-kosovo/2024/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-kosovo/2024/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/lillehammer-gp/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/gylne-gutuer/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/jelajah-cycling-series-minangkabau2/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/adriatica-ionica-race/2024/stage-1: single positional

In [13]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2024&month=10&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour octobre 2024")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[10] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(10)
driver.quit()

✅ 36 URLs trouvées pour octobre 2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is depre

✅ https://www.procyclingstats.com/race/tour-of-croatia/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/coppa-citta-di-san-daniele2/2024


...

✅ https://www.procyclingstats.com/race/veneto-classic/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/japan-cup/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is depre

✅ https://www.procyclingstats.com/race/vuelta-a-guatemala/2024
Total brut: 79 | Uniques: 36 | Manquantes: 0
❌ https://www.procyclingstats.com/race/gp-internacional-de-ciclismo-de-santa-catarina/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/gp-urubici-de-ciclismo/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/paris-bourges/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/grand-tour-de-ciclismo-de-sc/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-vendee/2024/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-beykoz/2024/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-beykoz/2024/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/international-azerbaijan-tou

In [14]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2024&month=11&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour novembre 2024")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[11] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(11)
driver.quit()

✅ 5 URLs trouvées pour novembre 2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/tour-do-rio/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/elite-road-caribbean-championships-itt/2024


...

✅ https://www.procyclingstats.com/race/elite-road-caribbean-championships-road-race/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/tour-de-okinawa/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is depre

✅ https://www.procyclingstats.com/race/vuelta-a-ecuador/2024
Total brut: 13 | Uniques: 5 | Manquantes: 0
❌ https://www.procyclingstats.com/race/tour-de-okinawa/2024/result: single positional indexer is out-of-bounds
✅ Mois 11: 13 fichiers sauvegardés


In [15]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2024&month=12&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour decembre 2024")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[12] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(12)
driver.quit()

✅ 6 URLs trouvées pour decembre 2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/tour-de-siak/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-india-itt/2024


...

✅ https://www.procyclingstats.com/race/nc-taiwan/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-india-rr/2024


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_26744/3720620909.py:64: FutureWarning: Passing literal html to 'read_html' is depre

✅ https://www.procyclingstats.com/race/vuelta-internacional-a-costa-rica/2024
Total brut: 17 | Uniques: 6 | Manquantes: 0
✅ Mois 12: 17 fichiers sauvegardés
